# M16 - Vision v2: real child handwriting + shapes
Retrains ONLY the vision tower (the "eyes"); the language model is untouched, so tutoring cannot change.
Data: the original 1,060 vision rows + generated shapes (exact labels) + generated pencil worksheets whose answers
are often WRONG or misspelled (the target is what is written - teaches "don't correct the child") + the 9 training
worksheets with reference labels (augmented). The 3 held-out worksheets are never trained on.
Output: `models/geeta-4b/mmproj-geeta-4b-v2-f16.gguf` (same Q4 language model). v2 is served on :8081 next to v1
and compared on every test; the live app is switched only if v2 wins with no regression.
Children's data stays under `data/raw/` (git-ignored).

In [ ]:
# ---- setup ---------------------------------------------------------------
import json, os, sys, time, math, random, re, shutil, subprocess, difflib
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "PROJECT_BRIEF.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
OUT = ROOT / "outputs" / "m16"; OUT.mkdir(parents=True, exist_ok=True)
GEN = ROOT / "data" / "raw" / "vision_v2"; GEN.mkdir(parents=True, exist_ok=True)   # generated + child images (git-ignored)
HW = ROOT / "data" / "raw" / "handwriting_labels"
MLX_DIR = "models/Qwen3.5-4B-MLX-8bit"
V1_HF = ROOT / "models" / "geeta-4b-hf"
V2_HF = ROOT / "models" / "geeta-4b-v2-hf"
GGUF_DIR = ROOT / "models" / "geeta-4b"
MMPROJ_V2 = GGUF_DIR / "mmproj-geeta-4b-v2-f16.gguf"
LLAMA = ROOT / "third_party" / "llama.cpp"
HW_PROMPT = ("This is a page of a child's maths homework. Write down ONLY what the child wrote by hand in pencil "
             "(answers, working, words) exactly as written, keeping their spelling mistakes. Ignore all printed text "
             "and the teacher's red marks. One answer per line.")
def done(f): return (OUT / f".{f}.done").exists()
def mark(f, p=None): (OUT / f".{f}.done").write_text(json.dumps({"ts": time.time(), "p": p}))
def log(name, obj): (OUT / name).write_text(json.dumps(obj, indent=1, default=str)); print("[m16] wrote", name)
def row(img, q, a): return {"messages": [{"role": "user", "content": [{"type": "image", "path": str(img)}, {"type": "text", "text": q}]},
                                          {"role": "assistant", "content": [{"type": "text", "text": a}]}]}
print("root", ROOT)

In [ ]:
# ---- A: generated shapes with exact labels --------------------------------
from PIL import Image, ImageDraw, ImageFilter, ImageFont
SH = GEN / "shapes"
def _reg(n, rot=0.0): return [(math.cos(rot + 2 * math.pi * i / n), math.sin(rot + 2 * math.pi * i / n)) for i in range(n)]
def _rotp(pts, a): c, s = math.cos(a), math.sin(a); return [(x * c - y * s, x * s + y * c) for x, y in pts]
def _clear_corners(pts, min_turn=22):
    """Every corner must visibly turn (no nearly-straight 'corners' a child couldn't count)."""
    n = len(pts)
    for i in range(n):
        (ax, ay), (bx, by), (cx, cy) = pts[i - 1], pts[i], pts[(i + 1) % n]
        a1, a2 = math.atan2(by - ay, bx - ax), math.atan2(cy - by, cx - bx)
        if abs(math.degrees((a2 - a1 + math.pi) % (2 * math.pi) - math.pi)) < min_turn:
            return False
    return True


REG_NAMES = {3: "triangle", 4: "square", 5: "pentagon", 6: "hexagon", 7: "heptagon", 8: "octagon"}
IRR_NAMES = {3: "triangle", 5: "pentagon", 6: "hexagon", 7: "heptagon", 8: "octagon"}

def make_shape(rng):
    k = rng.choices(["reg", "irr", "rect", "rhombus", "kite", "trap", "para", "rtri", "isotri", "star", "cross",
                     "chevron", "arrow", "house", "L", "circle", "oval"],
                    [16, 12, 7, 5, 4, 5, 4, 4, 4, 7, 4, 5, 4, 5, 3, 4, 3])[0]
    a = rng.uniform(0, 2 * math.pi)
    if k == "reg":
        n = rng.choice([3, 4, 5, 6, 7, 8]); return _reg(n, a), dict(sides=n, name=REG_NAMES[n], sym=n)
    if k == "irr":
        n = rng.choice([3, 5, 6, 7, 8]); angs = sorted(rng.uniform(0, 2 * math.pi) for _ in range(n))
        while min((angs[(i + 1) % n] - angs[i]) % (2 * math.pi) for i in range(n)) < 0.35:
            angs = sorted(rng.uniform(0, 2 * math.pi) for _ in range(n))
        while True:
            pts = [(rng.uniform(0.7, 1) * math.cos(t), rng.uniform(0.7, 1) * math.sin(t)) for t in angs]
            if _clear_corners(pts): break
            angs = sorted(rng.uniform(0, 2 * math.pi) for _ in range(n))
        return pts, dict(sides=n, name=IRR_NAMES[n], sym=None)
    if k == "rect":
        w, h = 1, rng.choice([0.45, 0.55, 0.65, 0.75]); return _rotp([(-w, -h), (w, -h), (w, h), (-w, h)], a if rng.random() < .5 else 0), dict(sides=4, name="rectangle", sym=2)
    if k == "rhombus":
        h = rng.uniform(0.45, 0.75); return _rotp([(0, -1), (h, 0), (0, 1), (-h, 0)], a if rng.random() < .4 else 0), dict(sides=4, name="rhombus", sym=2)
    if k == "kite":
        return _rotp([(0, -1), (0.6, -0.35), (0, 1), (-0.6, -0.35)], a if rng.random() < .3 else 0), dict(sides=4, name="kite", sym=1)
    if k == "trap":
        t = rng.uniform(0.35, 0.7); return [(-t, -0.7), (t, -0.7), (1, 0.7), (-1, 0.7)] if rng.random() < .5 else [(-1, -0.7), (1, -0.7), (t, 0.7), (-t, 0.7)], dict(sides=4, name="trapezium", sym=1)
    if k == "para":
        o = rng.uniform(0.3, 0.6); return [(-1 + o, -0.6), (1, -0.6), (1 - o, 0.6), (-1, 0.6)], dict(sides=4, name="parallelogram", sym=0)
    if k == "rtri":
        return _rotp([(-1, 0.7), (0.8, 0.7), (-1, -0.9)], a), dict(sides=3, name="triangle", sym=0)
    if k == "isotri":
        return [(0, -1), (0.6, 0.8), (-0.6, 0.8)], dict(sides=3, name="triangle", sym=1)
    if k == "star":
        p = rng.choice([5, 5, 5, 6]); pts = []
        for i in range(2 * p):
            r = 1 if i % 2 == 0 else 0.42; t = -math.pi / 2 + math.pi * i / p; pts.append((r * math.cos(t), r * math.sin(t)))
        return pts, dict(sides=2 * p, name="star", sym=p)
    if k == "cross":
        q = 0.33; return [(-q, -1), (q, -1), (q, -q), (1, -q), (1, q), (q, q), (q, 1), (-q, 1), (-q, q), (-1, q), (-1, -q), (-q, -q)], dict(sides=12, name="cross", sym=4)
    if k == "chevron":
        return [(-1, -0.6), (0.4, -0.6), (1, 0), (0.4, 0.6), (-1, 0.6), (-0.4, 0)], dict(sides=6, name=None, sym=1)
    if k == "arrow":
        return _rotp([(-1, -0.25), (0.2, -0.25), (0.2, -0.6), (1, 0), (0.2, 0.6), (0.2, 0.25), (-1, 0.25)], a if rng.random() < .5 else 0), dict(sides=7, name="arrow", sym=1)
    if k == "house":
        return [(-0.8, 1), (0.8, 1), (0.8, -0.2), (0, -1), (-0.8, -0.2)], dict(sides=5, name="pentagon", sym=1)
    if k == "L":
        return _rotp([(-0.8, -1), (-0.2, -1), (-0.2, 0.4), (0.8, 0.4), (0.8, 1), (-0.8, 1)], a), dict(sides=6, name=None, sym=None)
    if k == "circle":
        return "circle", dict(sides=0, name="circle", sym=None)
    return "oval", dict(sides=0, name="oval", sym=None)

def _wobble(pts, rng, amp):
    out = []
    for i in range(len(pts)):
        (x0, y0), (x1, y1) = pts[i], pts[(i + 1) % len(pts)]
        m = 6
        for j in range(m):
            t = j / m; nx, ny = -(y1 - y0), (x1 - x0); L = math.hypot(nx, ny) or 1
            d = rng.gauss(0, amp) if j else 0
            out.append((x0 + (x1 - x0) * t + nx / L * d, y0 + (y1 - y0) * t + ny / L * d))
    return out

def draw_shape(dr, shape, cx, cy, s, rng, ink, lw, fill):
    if isinstance(shape, str):
        rx, ry = (s, s) if shape == "circle" else (s, s * rng.uniform(0.5, 0.7))
        dr.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], outline=ink, width=lw, fill=fill); return
    pts = [(cx + x * s, cy + y * s) for x, y in shape]
    pts = _wobble(pts, rng, rng.uniform(0, 1.2))
    if fill: dr.polygon(pts, fill=fill)
    dr.line(pts + [pts[0]], fill=ink, width=lw, joint="curve")

def _canvas(rng, w, h):
    t = rng.randint(232, 255); return Image.new("RGB", (w, h), (t, t - rng.randint(0, 6), t - rng.randint(0, 14)))

def _finish(im, rng, path):
    if rng.random() < 0.5: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.2, 1.1)))
    if rng.random() < 0.3: im = im.rotate(rng.uniform(-4, 4), expand=False, fillcolor=im.getpixel((2, 2)))
    im.save(path, quality=rng.randint(45, 92))

Q_SIDES = "How many sides does this shape have? Answer with a number."
Q_CORN = "How many corners does this shape have? Answer with a number."
Q_SEG = "Write the number of segments and angles for this figure."
Q_NAME = "What is the name of this shape? Answer with one word."
Q_SYM = "How many lines of symmetry does this shape have? Answer with a number."
Q_DESC = "Describe this shape: its name, how many sides and how many corners."

def single(rng, path):
    S = rng.randint(288, 448); im = _canvas(rng, S, S); dr = ImageDraw.Draw(im)
    shape, info = make_shape(rng)
    ink = rng.choice([(20, 20, 20), (60, 60, 60), (90, 90, 95), (25, 35, 110)])
    fill = rng.choice([None] * 7 + [(40, 40, 40), (120, 120, 120), (200, 200, 200)])
    draw_shape(dr, shape, S / 2 + rng.uniform(-S * .06, S * .06), S / 2 + rng.uniform(-S * .06, S * .06), S * rng.uniform(.27, .4), rng, ink, rng.randint(2, 6), fill)
    if rng.random() < .2:
        for _ in range(rng.randint(1, 3)):
            x, y = rng.uniform(0, S), rng.uniform(0, S); dr.line([(x, y), (x + rng.uniform(-30, 30), y + rng.uniform(-10, 10))], fill=(150, 150, 150), width=1)
    _finish(im, rng, path); n = info["sides"]
    qa = [(Q_SIDES, str(n)), (Q_CORN, str(n))]
    if n: qa.append((Q_SEG, f"segments: {n}, angles: {n}"))
    if info["name"]: qa.append((Q_NAME, info["name"]))
    if info["sym"] is not None: qa.append((Q_SYM, str(info["sym"])))
    qa.append((Q_DESC, f"{info['name'] or 'shape'}: {n} sides, {n} corners"))
    return qa, info

def row_of_shapes(rng, path):
    k = rng.randint(3, 5); W, H = 120 * k + 40, 200; im = _canvas(rng, W, H); dr = ImageDraw.Draw(im); names = []
    for i in range(k):
        while True:
            shape, info = make_shape(rng)
            if info["name"] and info["name"] not in ("arrow", "cross", "heptagon", "parallelogram", "kite"): break
        draw_shape(dr, shape, 80 + 120 * i, H / 2, 45, rng, (20, 20, 20), rng.randint(2, 4), None); names.append(info["name"])
    _finish(im, rng, path)
    return [("Write the name of each shape, from left to right.", ", ".join(names))]

def count_scene(rng, path):
    W = H = 448; im = _canvas(rng, W, H); dr = ImageDraw.Draw(im)
    target = rng.choice(["triangle", "square", "circle", "star"]); n = rng.randint(1, 9)
    cells = [(c, r) for c in range(4) for r in range(4)]; rng.shuffle(cells)
    proto = {"triangle": _reg(3, -math.pi / 2), "square": _reg(4, math.pi / 4), "circle": "circle",
             "star": [((1 if i % 2 == 0 else .42) * math.cos(-math.pi / 2 + math.pi * i / 5), (1 if i % 2 == 0 else .42) * math.sin(-math.pi / 2 + math.pi * i / 5)) for i in range(10)]}
    others = [x for x in proto if x != target]
    for i, (c, r) in enumerate(cells[: n + rng.randint(0, 4)]):
        kind = target if i < n else rng.choice(others)
        draw_shape(dr, proto[kind], 56 + 112 * c + rng.uniform(-12, 12), 56 + 112 * r + rng.uniform(-12, 12), rng.uniform(22, 36), rng, (25, 25, 25), 3, None)
    _finish(im, rng, path)
    return [(f"How many {target}s are in this picture? Answer with a number.", str(n))]

def build_shapes(split, n_single, n_row, n_count, seed, qs_per=2):
    rng = random.Random(seed); (SH / split).mkdir(parents=True, exist_ok=True); rows = []
    for i in range(n_single):
        p = SH / split / f"s{i:05d}.jpg"; qa, _ = single(rng, p)
        for q, a in rng.sample(qa, min(qs_per, len(qa))): rows.append(row(p.relative_to(ROOT), q, a))
    for i in range(n_row):
        p = SH / split / f"r{i:05d}.jpg"; rows += [row(p.relative_to(ROOT), q, a) for q, a in row_of_shapes(rng, p)]
    for i in range(n_count):
        p = SH / split / f"c{i:05d}.jpg"; rows += [row(p.relative_to(ROOT), q, a) for q, a in count_scene(rng, p)]
    return rows

if done("A"):
    print("A done")
else:
    tr = build_shapes("train", 1300, 250, 400, seed=1601)
    te = build_shapes("test", 300, 40, 60, seed=9901, qs_per=1)
    (GEN / "shapes_train.jsonl").write_text("\n".join(json.dumps(r) for r in tr) + "\n")
    (GEN / "shapes_test.jsonl").write_text("\n".join(json.dumps(r) for r in te) + "\n")
    mark("A", {"train": len(tr), "test": len(te)}); print("shapes train", len(tr), "test", len(te))

In [ ]:
# ---- B: generated pencil worksheets (answers often wrong / misspelled) -------
FONTS = ["/System/Library/Fonts/Supplemental/Bradley Hand Bold.ttf", "/System/Library/Fonts/Noteworthy.ttc",
         "/System/Library/Fonts/MarkerFelt.ttc", "/System/Library/Fonts/Supplemental/Chalkboard.ttc",
         "/System/Library/Fonts/Supplemental/ChalkboardSE.ttc", "/System/Library/Fonts/Supplemental/Comic Sans MS.ttf",
         "/System/Library/Fonts/Supplemental/Chalkduster.ttf"]
FONTS = [f for f in FONTS if os.path.exists(f)]
PRINT = "/System/Library/Fonts/Supplemental/Arial.ttf"
MISSPELL = {"hexagon": ["hexigon", "hexigan", "hexagan"], "pentagon": ["pentegon", "pentigon"], "octagon": ["octogon", "octigon"],
            "triangle": ["triangl", "tryangle"], "rectangle": ["rectangl", "rectangel"], "chocolates": ["chocolets", "chocolats"],
            "buttons": ["bottons", "butons"], "hundred": ["hurnd", "hundrd", "hunderd"], "eight": ["eght", "eigt"],
            "thirty": ["therty", "thirdy"], "sandwiches": ["sandweches", "sandwhiches"], "ladder": ["lader", "Lader"],
            "Thursday": ["Thursdy", "thursday"], "Wednesday": ["wednesday", "Wensday"], "dollars": ["dollers", "dolars"],
            "packets": ["packits"], "nappies": ["napies"], "balloons": ["baloons"], "squares": ["sqares"]}
NOUNS = ["apples", "books", "eggs", "kids", "packets", "boxes", "lollies", "chocolates", "buttons", "days", "hours", "legs",
         "cards", "presents", "nappies", "balloons", "cakes", "marbles", "stickers", "dollars", "groups", "squares"]

def kid_answer(rng, true):
    """What a Year-2 child might really write: usually right, sometimes a slip."""
    r = rng.random()
    if r < 0.62 or true < 3: return true
    return rng.choice([true + 1, true - 1, true + 10, true - 10, int(str(true)[::-1]) if true > 9 else true + 2, true + rng.randint(2, 9)])

def misspell(rng, word):
    return rng.choice(MISSPELL[word]) if word in MISSPELL and rng.random() < 0.45 else word

class Pencil:
    def __init__(self, rng):
        self.rng = rng; self.font = rng.choice(FONTS); self.size = rng.randint(26, 40)
        g = rng.randint(55, 125); self.col = (g, g, g + rng.randint(0, 12))
    def write(self, im, x, y, text):
        dr = ImageDraw.Draw(im); rng = self.rng
        for ch in text:
            sz = max(14, int(self.size * rng.uniform(0.82, 1.18)))
            f = ImageFont.truetype(self.font, sz)
            tile = Image.new("L", (sz * 2, sz * 2), 0); ImageDraw.Draw(tile).text((sz // 3, sz // 3), ch, font=f, fill=255)
            tile = tile.rotate(rng.uniform(-12, 12), resample=Image.BICUBIC)
            col = Image.new("RGB", tile.size, self.col)
            im.paste(col, (int(x - sz // 3), int(y - sz // 3 + rng.uniform(-3, 3))), tile.point(lambda v: int(v * rng.uniform(0.55, 0.95))))
            x += (dr.textlength(ch, font=f) if ch != " " else sz * 0.45) * rng.uniform(0.95, 1.15)
        return x

def teacher_mark(im, rng, x, y):
    dr = ImageDraw.Draw(im); red = (210, 40, 50)
    if rng.random() < 0.8: dr.line([(x, y + 10), (x + 10, y + 22), (x + 30, y - 6)], fill=red, width=3)
    else: dr.line([(x, y), (x + 22, y + 22)], fill=red, width=3); dr.line([(x + 22, y), (x, y + 22)], fill=red, width=3)

def gen_page(rng, path):
    W, H = 1000, 1400; im = _canvas(rng, W, H); dr = ImageDraw.Draw(im)
    pf = ImageFont.truetype(PRINT, 30); small = ImageFont.truetype(PRINT, 22)
    dr.text((80, 50), rng.choice(["Speed and Accuracy", "Problem Solving", "Try these!", "Homework", "Multiplication"]), font=ImageFont.truetype(PRINT, 44), fill=(30, 30, 30))
    pen = Pencil(rng); lines = []
    layout = rng.choice(["grid", "grid", "problems"])
    if layout == "grid":
        cols, rows = rng.choice([(3, 10), (2, 12), (3, 8), (2, 9)])
        ops = rng.choice(["x", "x", "+", "-"])
        for r in range(rows):
            for c in range(cols):
                a, b = rng.randint(1, 12), rng.randint(1, 12)
                if ops == "+": a, b = rng.randint(5, 99), rng.randint(2, 60)
                if ops == "-": a = rng.randint(10, 99); b = rng.randint(1, a)
                true = a * b if ops == "x" else (a + b if ops == "+" else a - b)
                x0, y0 = 70 + c * (W - 100) // cols, 150 + r * (H - 220) // rows
                q = f"{a} {ops} {b} = "; dr.text((x0, y0), q, font=pf, fill=(20, 20, 20))
                bx = x0 + dr.textlength(q, font=pf) + 6; dr.line([(bx, y0 + 38), (bx + 90, y0 + 38)], fill=(40, 40, 40), width=2)
                if rng.random() < 0.06: continue                               # left blank
                ans = str(kid_answer(rng, true)); end = pen.write(im, bx + 8, y0 - 4, ans); lines.append(ans)
                if rng.random() < 0.7: teacher_mark(im, rng, end + 8, y0)
    else:
        y = 160
        for i in range(rng.randint(3, 5)):
            a, b = rng.randint(2, 12), rng.randint(2, 12); noun = rng.choice(NOUNS); true = a * b
            dr.text((70, y), f"{i + 1}. There are {a} {noun} in each box. How many {noun} in {b} boxes?", font=small, fill=(20, 20, 20))
            dr.text((90, y + 50), "Number sentence: ________________", font=small, fill=(20, 20, 20))
            dr.text((90, y + 100), "There are ________________", font=small, fill=(20, 20, 20))
            ans = kid_answer(rng, true); sent = f"{a}x{b}={ans}" if rng.random() < .7 else f"{b} x {a} = {ans}"
            word = f"{ans} {misspell(rng, noun)}"
            pen.write(im, 290, y + 34, sent); pen.write(im, 250, y + 84, word); lines += [sent, word]
            if rng.random() < 0.7: teacher_mark(im, rng, 600, y + 40)
            y += 230
        for shape in rng.sample(list(MISSPELL)[:5], rng.randint(0, 2)):
            dr.text((70, y), "Name this shape:", font=small, fill=(20, 20, 20)); w = misspell(rng, shape)
            pen.write(im, 290, y - 14, w); lines.append(w); y += 70
    if rng.random() < 0.4:
        t = f"{rng.randint(1, 12)}:{rng.randint(0, 59):02d}"; dr.text((640, 60), "Time:", font=pf, fill=(20, 20, 20)); pen.write(im, 740, 52, t); lines.insert(0, t)
    _finish(im, rng, path)
    return "\n".join(lines)

if done("B"):
    print("B done")
else:
    rng = random.Random(1602); d = GEN / "pencil"; d.mkdir(parents=True, exist_ok=True); rows = []
    for i in range(700):
        p = d / f"p{i:04d}.jpg"; rows.append(row(p.relative_to(ROOT), HW_PROMPT, gen_page(rng, p)))
    (GEN / "pencil_train.jsonl").write_text("\n".join(json.dumps(r) for r in rows) + "\n")
    mark("B", {"pages": len(rows)}); print("pencil pages", len(rows))

In [ ]:
# ---- C: the child's own pages (9 training worksheets, augmented x4) --------
from PIL import ImageEnhance
LAB = {k: v for k, v in json.loads((HW / "reference_labels.json").read_text()).items() if k != "_notes"}
if done("C"):
    print("C done")
else:
    rng = random.Random(1603); d = GEN / "child_train"; d.mkdir(parents=True, exist_ok=True)
    pages = {}
    for k, v in LAB.items():
        if not v["heldout"]: pages.setdefault(v["img"], []).append((k, v))
    rows, n_pages = [], 0
    for img, items in sorted(pages.items()):
        if any(v["status"] == "unreadable" for _, v in items):
            continue                                 # a page with an unreadable answer can't have a complete target
        lines = [v["text"] for _, v in items if v["status"] == "checked" and v["text"].strip()]
        target = "\n".join(lines) if lines else "(nothing handwritten)"
        base = Image.open(HW / img).convert("RGB"); n_pages += 1
        for j in range(4):
            im = base.copy()
            if j:
                im = im.rotate(rng.uniform(-3, 3), expand=False, fillcolor=(245, 245, 245))
                im = ImageEnhance.Brightness(im).enhance(rng.uniform(0.85, 1.12))
                im = ImageEnhance.Contrast(im).enhance(rng.uniform(0.8, 1.25))
                s = rng.uniform(0.8, 1.15); im = im.resize((int(im.width * s), int(im.height * s)))
                if rng.random() < .5: im = im.filter(ImageFilter.GaussianBlur(rng.uniform(0.2, 0.9)))
            p = d / f"{Path(img).stem}_{j}.jpg"; im.save(p, quality=rng.randint(60, 92))
            rows.append(row(p.relative_to(ROOT), HW_PROMPT, target))
    (GEN / "child_train.jsonl").write_text("\n".join(json.dumps(r) for r in rows) + "\n")
    mark("C", {"pages": n_pages, "rows": len(rows)}); print("child pages", n_pages, "rows", len(rows))

In [ ]:
# ---- D: train the vision tower (language model frozen) ---------------------
if done("D"):
    print("D done")
else:
    import mlx.core as mx, mlx.nn as nn, mlx.optimizers as optim, mlx_vlm
    from mlx.utils import tree_flatten
    from mlx_vlm import utils as vutils
    from mlx_vlm.trainer.utils import grad_checkpoint, get_learning_rate
    rows = []
    for f, reps in ((ROOT / "data/splits/train_vision.jsonl", 1), (GEN / "shapes_train.jsonl", 1),
                    (GEN / "pencil_train.jsonl", 1), (GEN / "child_train.jsonl", 2)):
        rr = [json.loads(l) for l in open(f) if l.strip()]; rows += rr * reps; print(f.name, len(rr), "x", reps)
    random.Random(1604).shuffle(rows)
    model, processor = vutils.load(MLX_DIR); model.train()
    MAXSIDE = 1100
    def prep(r):
        msgs = r["messages"]; img = Image.open(ROOT / msgs[0]["content"][0]["path"]).convert("RGB")
        if max(img.size) > MAXSIDE:
            s = MAXSIDE / max(img.size); img = img.resize((int(img.width * s), int(img.height * s)))
        pf = mlx_vlm.apply_chat_template(processor, model.config, msgs, add_generation_prompt=False, num_images=1)
        pp = mlx_vlm.apply_chat_template(processor, model.config, msgs[:1], add_generation_prompt=True, num_images=1)
        a = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[pf]); b = mlx_vlm.prepare_inputs(processor, images=[img], prompts=[pp])
        off = b["input_ids"].shape[1]
        assert bool((a["input_ids"][:, :off] == b["input_ids"]).all()), "prefix misaligned"
        return a["input_ids"], a.get("pixel_values"), a.get("image_grid_thw"), off
    model.freeze(); model.vision_tower.unfreeze(); grad_checkpoint(model.vision_tower.blocks[0])
    opt = optim.Adam(learning_rate=1e-5)
    def loss_fn(m_, ids, pv, grid, off):
        logits = m_(ids, pv, None, None, image_grid_thw=grid).logits[:, :-1]; tg = ids[:, 1:]
        steps = mx.arange(1, tg.shape[1] + 1); mask = steps >= off
        ce = nn.losses.cross_entropy(logits, tg) * mask
        return ce.astype(mx.float32).sum() / mx.maximum(mask.sum(), 1)
    vg = nn.value_and_grad(model, loss_fn)
    ITERS = len(rows); hist = []; t0 = time.time(); skipped = 0
    for it in range(1, ITERS + 1):
        opt.learning_rate = get_learning_rate(ITERS, it, 100, 1e-5, 1e-6)
        try:
            ids, pv, grid, off = prep(rows[it - 1])
        except Exception as e:
            skipped += 1; continue
        lv, g = vg(model, ids, pv, grid, off); opt.update(model, g); mx.eval(model.parameters(), opt.state)
        hist.append(float(lv))
        if it % 50 == 0 or it == 1:
            el = time.time() - t0
            msg = f"[vision v2] {it}/{ITERS} loss {sum(hist[-50:]) / len(hist[-50:]):.4f} | {el / 60:.1f} min, eta {el / it * (ITERS - it) / 60:.0f} min"
            print(msg, flush=True)
            with open(OUT / "progress.txt", "a") as pf: pf.write(msg + "\n")
    vdir = OUT / "vision_tower_v2"; vdir.mkdir(exist_ok=True)
    mx.save_safetensors(str(vdir / "adapters.safetensors"), dict(tree_flatten(model.vision_tower.parameters())))
    n = 50; first, last = sum(hist[:n]) / n, sum(hist[-n:]) / n
    log("train_report.json", {"iters": ITERS, "skipped": skipped, "minutes": round((time.time() - t0) / 60, 1),
                              "first_loss": first, "last_loss": last, "loss_curve": [round(x, 4) for x in hist[::20]]})
    assert last < first, "loss did not drop"
    mark("D", {"first": first, "last": last})

In [ ]:
# ---- E: v2 vision weights -> HF dir -> mmproj GGUF (language model unchanged) ---
if done("E"):
    print("E done")
else:
    import mlx.core as mx, numpy as np
    trained = mx.load(str(OUT / "vision_tower_v2" / "adapters.safetensors"))
    v1 = dict(mx.load(str(V1_HF / "model.safetensors")))
    n_rep = 0
    for k, v in trained.items():
        hk = "model.visual." + k
        a = np.asarray(v.astype(mx.float16))
        if hk.endswith("patch_embed.proj.weight") and a.ndim == 5:
            a = np.transpose(a, (0, 4, 1, 2, 3))
        assert hk in v1, f"unknown key {hk}"
        assert tuple(v1[hk].shape) == a.shape, f"shape {hk} {v1[hk].shape} vs {a.shape}"
        v1[hk] = mx.array(a.astype(np.float16)); n_rep += 1
    n_vis = sum(1 for k in v1 if k.startswith("model.visual."))
    print("replaced", n_rep, "of", n_vis, "vision tensors")
    assert n_rep == n_vis, "not every vision tensor was replaced"
    V2_HF.mkdir(parents=True, exist_ok=True)
    mx.save_safetensors(str(V2_HF / "model.safetensors"), v1)
    for fn in os.listdir(V1_HF):
        if fn != "model.safetensors": shutil.copy2(V1_HF / fn, V2_HF / fn)
    del v1
    r = subprocess.run([sys.executable, str(LLAMA / "convert_hf_to_gguf.py"), str(V2_HF), "--mmproj", "--outtype", "f16",
                        "--outfile", str(MMPROJ_V2)], cwd=LLAMA, capture_output=True, text=True)
    print(r.stdout[-400:], r.stderr[-800:] if r.returncode else "")
    assert MMPROJ_V2.exists() and MMPROJ_V2.stat().st_size > 100_000_000, "mmproj v2 not produced"
    mark("E", {"mmproj_gb": round(MMPROJ_V2.stat().st_size / 2**30, 3)}); print("mmproj v2", MMPROJ_V2)

In [ ]:
# ---- F: v1 (:8080, live) vs v2 (:8081) on every vision test ----------------
nb = json.load(open(ROOT / "tools/_nbmodule.ipynb")); exec("".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")); bootstrap(ROOT)
from geeta.model_client import ModelClient
from geeta.worksheet import split_worksheet
import urllib.request
def up(url):
    try: urllib.request.urlopen(url + "/health", timeout=3); return True
    except Exception: return False
V2URL = "http://127.0.0.1:8081"
if not up(V2URL):
    srv = subprocess.Popen([str(LLAMA / "llama-server"), "--model", str(GGUF_DIR / "geeta-4b-q4_k_m.gguf"), "--mmproj", str(MMPROJ_V2),
                            "--host", "127.0.0.1", "--port", "8081", "-c", "8192", "-ngl", "99"],
                           stdout=open(OUT / "server_v2.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(90):
        if up(V2URL): break
        time.sleep(2)
assert up(V2URL), "v2 server did not start"
norm = lambda s: re.sub(r"\s+", " ", (s or "").lower().replace("×", "x").replace("÷", "/")).strip()
def cer(gt, pr):
    import difflib; return difflib.SequenceMatcher(None, norm(gt), norm(pr)).ratio()
SYN = {"rhombus": ["rhombus", "diamond"], "trapezium": ["trapezium", "trapezoid"], "cross": ["cross", "plus"]}
def shape_ok(q, truth, out):
    o = norm(out)
    if re.fullmatch(r"-?\d+", truth): m = re.search(r"\d+", o); return bool(m) and m.group(0) == truth
    if truth.startswith("segments"): return re.findall(r"\d+", o)[:2] == re.findall(r"\d+", truth)
    if "," in truth: return [w.strip() for w in o.split(",")] == [w.strip() for w in truth.split(",")]
    return any(w in o for w in SYN.get(truth, [truth]))
REAL_SHAPES = [("p9_cross", 12, ["cross", "plus"]), ("p9_diamond", 4, ["rhombus", "diamond", "square"]), ("p9_house", 5, ["pentagon"]),
               ("p9_trapezium", 4, ["trapezium", "trapezoid"]), ("p9_chevron", 6, None), ("p9_star", 10, ["star"]),
               ("wk3_square", 4, ["square"]), ("wk3_hexagon", 6, ["hexagon"]), ("wk3_triangle", 3, ["triangle"]),
               ("wk3_circle", 0, ["circle"]), ("wk3_rectangle", 4, ["rectangle"])]
PHOTO_TRUTH = {"booklet_p8_division": {"1": "8 each, 0 left over", "2": "4 each, 6 left over", "3": "4 each, 1 left over"},
               "booklet_p7_problems": {"1": 6, "2": "4 left over", "3": 9, "4": 7},
               "booklet_p9_shapes": {str(i): "picture" for i in range(1, 9)},
               "booklet_p10_times": dict({str(i + 1): v for i, v in enumerate([9, 8, 24, 30, 54, 7, 12, 72, 50, 32, 56, 12, 49, 48, 16, 63, 35, 40, 21, 36])}, **{str(i): "picture" for i in range(21, 26)}),
               "booklet_p11_times3": {str(i + 1): v for i, v in enumerate([3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36, 90, 0, 150, 60, 45, 120, 75, 300])}}
def evaluate(url):
    m = ModelClient(url); res = {}
    te = [json.loads(l) for l in open(GEN / "shapes_test.jsonl")]
    ok = 0
    for r in te:
        q = r["messages"][0]["content"][1]["text"]; a = r["messages"][1]["content"][0]["text"]
        ok += shape_ok(q, a, m.chat_image(ROOT / r["messages"][0]["content"][0]["path"], q, temperature=0.0, max_tokens=40))
    res["shapes_generated_test"] = round(ok / len(te), 4)
    s_ok = n_ok = n_n = 0
    for name, sides, names in REAL_SHAPES:
        p = ROOT / "data/raw/real_shapes" / f"{name}.png"
        o = norm(m.chat_image(p, "How many sides does this shape have? Answer with a number.", temperature=0.0, max_tokens=8))
        s_ok += (re.findall(r"\d+", o)[:1] == [str(sides)])
        if names:
            n_n += 1; o = norm(m.chat_image(p, "What is the name of this shape? Answer with one word.", temperature=0.0, max_tokens=8))
            n_ok += any(w in o for w in names)
    res["real_shapes_sides"] = f"{s_ok}/{len(REAL_SHAPES)}"; res["real_shapes_names"] = f"{n_ok}/{n_n}"
    man = json.load(open(ROOT / "data/eval/handwriting_manifest.json"))["eval"]
    res["synthetic_handwriting_char_acc"] = round(sum(cer(x["transcription"], m.ocr_page(ROOT / x["image"])) for x in man) / len(man), 4)
    pages = {}
    for v in LAB.values():
        if v["heldout"] and v["status"] == "checked": pages.setdefault(v["img"], []).append(v)
    found = tot = 0
    for img, labs in sorted(pages.items()):
        o = norm(m.chat_image(HW / img, HW_PROMPT, temperature=0.0, max_tokens=900))
        for v in labs:
            for part in v["text"].split("\n"):
                g = norm(part)
                if g: tot += 1; found += re.search(r"(?<![\w$.])" + re.escape(g) + r"(?![\w.])", o) is not None
    res["real_child_handwriting_exact"] = round(found / tot, 4); res["real_child_handwriting_n"] = tot
    good = total = 0
    for name, truth in PHOTO_TRUTH.items():
        its = split_worksheet(m.ocr_page(ROOT / "data/raw/real_photos" / f"{name}.jpg"))
        got = {i.label: (i.kind if i.kind == "picture" else (i.meta["problem"].answer if i.meta.get("problem") else "?")) for i in its}
        total += len(truth); good += sum(got.get(k) == v for k, v in truth.items())
    res["real_photo_pages_items"] = f"{good}/{total}"
    return res
if done("F"):
    print(json.loads((OUT / "compare.json").read_text()))
else:
    t0 = time.time(); r1 = evaluate("http://127.0.0.1:8080"); r2 = evaluate(V2URL)
    cmp = {"v1_live": r1, "v2": r2, "minutes": round((time.time() - t0) / 60, 1)}
    frac = lambda s: int(s.split("/")[0]) / int(s.split("/")[1])
    gate = (r2["real_child_handwriting_exact"] > r1["real_child_handwriting_exact"]
            and r2["shapes_generated_test"] > r1["shapes_generated_test"]
            and r2["synthetic_handwriting_char_acc"] >= r1["synthetic_handwriting_char_acc"] - 0.02
            and frac(r2["real_photo_pages_items"]) >= frac(r1["real_photo_pages_items"]) - 0.02)
    cmp["v2_better_no_regression"] = gate
    log("compare.json", cmp); print(json.dumps(cmp, indent=1)); mark("F", gate)